In [1]:
import open3d as o3d
import numpy as np
import os
import sys
from glob import glob

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')
from utils.write_output import write_array2txt

# # monkey patches visualization and provides helpers to load geometries
# sys.path.append('..')
# import open3d_tutorial as o3dtut
# # change to True if you want to interact with the visualization windows
# o3dtut.interactive = not "CI" in os.environ

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# Multiway registration
Multiway registration is the process of aligning multiple pieces of geometry in a global space. Typically, the input is a set of geometries (e.g., point clouds or RGBD images) $\{\mathbf{P}_{i}\}$. The output is a set of rigid transformations $\{\mathbf{T}_{i}\}$, so that the transformed point clouds $\{\mathbf{T}_{i}\mathbf{P}_{i}\}$ are aligned in the global space.

Open3D implements multiway registration via pose graph optimization. The backend implements the technique presented in [\[Choi2015\]](../reference.html#choi2015).

## Input
The first part of the tutorial code reads three point clouds from files. The point clouds are downsampled and visualized together. They are misaligned.

# Align bottom

In [13]:
data_dir = 'D:/REGISTRATION DATA/External/OLD Sampled External Scans/JB-448 26ft EXT/tank_0'
write_dir = os.path.join(data_dir, 'bottom_aligned')
if not os.path.exists(write_dir):
    os.mkdir(write_dir)
for i in range(88,97):     
    print(i) 
    filepath = glob(os.path.join(
        data_dir,
        '*{:03d}*.txt'.format(i)
    ))[0]
    pcd = o3d.io.read_point_cloud(filepath,format='xyz')
    # slide to z_min = 0
    points = np.asarray(pcd.points)
    points[:,2] = points[:,2] - points[:,2].min()
    pcd.points = o3d.utility.Vector3dVector(points)
    # save
    write_array2txt(
        os.path.join(
            write_dir, f'0{i}.txt'
        ),
        points
    )

88
89
90
91
92
93
94
95
96


# Load data

In [2]:

def load_point_clouds(path_,voxel_size=0.0):
    pcds = []
    for i in [57, 58]:#range(88,97): 
        filepath = glob(os.path.join(
            path_,
            '*{:03d}*.txt'.format(i)
        ))[0]
        print(filepath)
        pcd = o3d.io.read_point_cloud(filepath,format='xyz')

        pcd_down = pcd.voxel_down_sample(voxel_size=voxel_size)
        pcd_down.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.2,max_nn=10))
        pcds.append(pcd_down)
        # pcd.estimate_normals()
        # pcds.append(pcd)
    return pcds

In [3]:
voxel_size = 0.02
path_ = 'D:/REGISTRATION DATA/External/JB-483 70ft EXT/tank_1'
pcds_down = load_point_clouds(path_,voxel_size)
# for i in range(5):
#     pcds_down[i].estimate_normals()
# o3d.visualization.draw_geometries(pcds_down,
#                                   zoom=0.3412,
#                                   front=[0.4257, -0.2125, -0.8795],
#                                   lookat=[2.6172, 2.0475, 1.532],
#                                   up=[-0.0694, -0.9768, 0.2024])

D:/REGISTRATION DATA/External/JB-483 70ft EXT/tank_1\057.txt
D:/REGISTRATION DATA/External/JB-483 70ft EXT/tank_1\058.txt


## Pose graph
A pose graph has two key elements: nodes and edges. A node is a piece of geometry $\mathbf{P}_{i}$ associated with a pose matrix $\mathbf{T}_{i}$ which transforms $\mathbf{P}_{i}$ into the global space. The set $\{\mathbf{T}_{i}\}$ are the unknown variables to be optimized. `PoseGraph.nodes` is a list of `PoseGraphNode`. We set the global space to be the space of $\mathbf{P}_{0}$. Thus $\mathbf{T}_{0}$ is the identity matrix. The other pose matrices are initialized by accumulating transformation between neighboring nodes. The neighboring nodes usually have large overlap and can be registered with [Point-to-plane ICP](../pipelines/icp_registration.ipynb#point-to-plane-ICP).

A pose graph edge connects two nodes (pieces of geometry) that overlap. Each edge contains a transformation matrix $\mathbf{T}_{i,j}$ that aligns the source geometry $\mathbf{P}_{i}$ to the target geometry $\mathbf{P}_{j}$. This tutorial uses [Point-to-plane ICP](../pipelines/icp_registration.ipynb#point-to-plane-ICP) to estimate the transformation. In more complicated cases, this pairwise registration problem should be solved via [Global registration](global_registration.ipynb).

[\[Choi2015\]](../reference.html#choi2015) has observed that pairwise registration is error-prone. False pairwise alignments can outnumber correctly aligned pairs. Thus, they partition pose graph edges into two classes. **Odometry edges** connect temporally close, neighboring nodes. A local registration algorithm such as ICP can reliably align them. **Loop closure edges** connect any non-neighboring nodes. The alignment is found by global registration and is less reliable. In Open3D, these two classes of edges are distinguished by the `uncertain` parameter in the initializer of `PoseGraphEdge`.

In addition to the transformation matrix $\mathbf{T}_{i}$, the user can set an information matrix $\mathbf{\Lambda}_{i}$ for each edge. If $\mathbf{\Lambda}_{i}$ is set using function `get_information_matrix_from_point_clouds`, the loss on this pose graph edge approximates the RMSE of the corresponding sets between the two nodes, with a line process weight. Refer to Eq (3) to (9) in [\[Choi2015\]](../reference.html#choi2015) and [the Redwood registration benchmark](http://redwood-data.org/indoor/registration.html) for details.

The script creates a pose graph with three nodes and three edges. Among the edges, two of them are odometry edges (`uncertain = False`) and one is a loop closure edge (`uncertain = True`).

In [4]:
def pairwise_registration(source, target):
    print("Apply point-to-plane ICP")
    print("Coarse alignment")  
    # coarse alignment
    icp_coarse = o3d.pipelines.registration.registration_icp(
        source, target, max_correspondence_distance_coarse, np.identity(4),
        o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=100))
    # fine aligment  
    print("Fine alignment")  
    icp_fine = o3d.pipelines.registration.registration_icp(
        source, target, max_correspondence_distance_fine,
        icp_coarse.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=300))
        
    transformation_icp = icp_fine.transformation
    # get transformation matrix
    information_icp = o3d.pipelines.registration.get_information_matrix_from_point_clouds(
        source, target, max_correspondence_distance_fine,
        icp_fine.transformation)
    return transformation_icp, information_icp


def full_registration(pcds, max_correspondence_distance_coarse,
                      max_correspondence_distance_fine):
    pose_graph = o3d.pipelines.registration.PoseGraph()
    odometry = np.identity(4)
    pose_graph.nodes.append(o3d.pipelines.registration.PoseGraphNode(odometry))
    n_pcds = len(pcds)
    for source_id in range(n_pcds):
        for target_id in range(source_id + 1, n_pcds):
            transformation_icp, information_icp = pairwise_registration(
                pcds[source_id], pcds[target_id])
            print("Build o3d.pipelines.registration.PoseGraph")
            if target_id == source_id + 1:  # odometry case
                odometry = np.dot(transformation_icp, odometry)
                pose_graph.nodes.append(
                    o3d.pipelines.registration.PoseGraphNode(
                        np.linalg.inv(odometry)))
                pose_graph.edges.append(
                    o3d.pipelines.registration.PoseGraphEdge(source_id,
                                                             target_id,
                                                             transformation_icp,
                                                             information_icp,
                                                             uncertain=False))
            else:  # loop closure case
                pose_graph.edges.append(
                    o3d.pipelines.registration.PoseGraphEdge(source_id,
                                                             target_id,
                                                             transformation_icp,
                                                             information_icp,
                                                             uncertain=True))
    return pose_graph

In [9]:
print("Full registration ...")
max_correspondence_distance_coarse = 0.02#voxel_size * 15
max_correspondence_distance_fine = 0.005#voxel_size * 1.5
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    pose_graph = full_registration(pcds_down,
                                   max_correspondence_distance_coarse,
                                   max_correspondence_distance_fine)

Full registration ...
Apply point-to-plane ICP
Coarse alignment
[Open3D DEBUG] ICP Iteration #0: Fitness 0.0067, RMSE 0.0131
[Open3D DEBUG] Residual : 1.12e-04 (# of elements : 7020)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.0067, RMSE 0.0131
[Open3D DEBUG] Residual : 1.07e-04 (# of elements : 6974)
[Open3D DEBUG] ICP Iteration #2: Fitness 0.0067, RMSE 0.0132
[Open3D DEBUG] Residual : 1.06e-04 (# of elements : 6997)
[Open3D DEBUG] ICP Iteration #3: Fitness 0.0067, RMSE 0.0135
[Open3D DEBUG] Residual : 1.05e-04 (# of elements : 7015)
[Open3D DEBUG] ICP Iteration #4: Fitness 0.0067, RMSE 0.0135
[Open3D DEBUG] Residual : 1.03e-04 (# of elements : 6966)
[Open3D DEBUG] ICP Iteration #5: Fitness 0.0067, RMSE 0.0136
[Open3D DEBUG] Residual : 1.02e-04 (# of elements : 6977)
[Open3D DEBUG] ICP Iteration #6: Fitness 0.0068, RMSE 0.0137
[Open3D DEBUG] Residual : 1.03e-04 (# of elements : 7083)
[Open3D DEBUG] ICP Iteration #7: Fitness 0.0068, RMSE 0.0136
[Open3D DEBUG] Residual : 1.02e-04 (# of e

Open3D uses the function `global_optimization` to perform pose graph optimization. Two types of optimization methods can be chosen: `GlobalOptimizationGaussNewton` or `GlobalOptimizationLevenbergMarquardt`. The latter is recommended since it has better convergence property. Class `GlobalOptimizationConvergenceCriteria` can be used to set the maximum number of iterations and various optimization parameters.

Class `GlobalOptimizationOption` defines a couple of options. `max_correspondence_distance` decides the correspondence threshold. `edge_prune_threshold` is a threshold for pruning outlier edges. `reference_node` is the node id that is considered to be the global space.

In [10]:
print("Optimizing PoseGraph ...")
option = o3d.pipelines.registration.GlobalOptimizationOption(
    max_correspondence_distance=max_correspondence_distance_fine,
    edge_prune_threshold=0.25,
    reference_node=0)
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option)

Optimizing PoseGraph ...
[Open3D DEBUG] Validating PoseGraph - finished.
[Open3D DEBUG] [GlobalOptimizationLM] Optimizing PoseGraph having 2 nodes and 1 edges.
[Open3D DEBUG] Line process weight : 0.655850
[Open3D DEBUG] [Initial     ] residual : 1.307689e-26, lambda : 1.833313e+02
[Open3D DEBUG] Maximum coefficient of right term < 1.000000e-06
[Open3D DEBUG] [GlobalOptimizationLM] Optimizing PoseGraph having 2 nodes and 1 edges.
[Open3D DEBUG] Line process weight : 0.655850
[Open3D DEBUG] [Initial     ] residual : 1.307689e-26, lambda : 1.833313e+02
[Open3D DEBUG] Maximum coefficient of right term < 1.000000e-06
[Open3D DEBUG] CompensateReferencePoseGraphNode : reference : 0


The global optimization performs twice on the pose graph. The first pass optimizes poses for the original pose graph taking all edges into account and does its best to distinguish false alignments among uncertain edges. These false alignments have small line process weights, and they are pruned after the first pass. The second pass runs without them and produces a tight global alignment. In this example, all the edges are considered as true alignments, hence the second pass terminates immediately.

## Visualize optimization
The transformed point clouds are listed and visualized using `draw_geometries`.

In [12]:
print("Transform points and display")
for point_id in range(len(pcds_down)):
    print(pose_graph.nodes[point_id].pose)
    pcds_down[point_id].transform(pose_graph.nodes[point_id].pose)
o3d.visualization.draw_geometries(pcds_down,
                                  zoom=0.3412,
                                  front=[0.4257, -0.2125, -0.8795],
                                  lookat=[2.6172, 2.0475, 1.532],
                                  up=[-0.0694, -0.9768, 0.2024])

Transform points and display
[[ 1.00000000e+00 -7.21715084e-22  0.00000000e+00  0.00000000e+00]
 [-7.21508289e-22  1.00000000e+00  0.00000000e+00 -5.42101086e-20]
 [-1.35525272e-20  0.00000000e+00  1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
[[ 0.99961391  0.00231362 -0.02768893  0.69280625]
 [-0.00255626  0.99995862 -0.00873099  0.2652015 ]
 [ 0.02766759  0.0087984   0.99957846 -0.23322677]
 [ 0.          0.          0.          1.        ]]
[[ 0.99992079 -0.00527746 -0.01142681  0.26809139]
 [ 0.00509784  0.99986389 -0.015691    0.17682358]
 [ 0.01150807  0.01563151  0.99981159 -0.04023892]
 [ 0.          0.          0.          1.        ]]
[[ 0.99986155 -0.00980304 -0.01344566  0.38556028]
 [ 0.00984317  0.99994729  0.00292174 -0.97750253]
 [ 0.01341631 -0.00305368  0.99990533 -0.23019369]
 [ 0.          0.          0.          1.        ]]
[[ 9.93799074e-01 -1.01979344e-01 -4.43126840e-02  3.81312101e-01]
 [ 1.02123454e-01  9

## Make a combined point cloud
`PointCloud` has a convenience operator `+` that can merge two point clouds into a single one. In the code below, the points are uniformly resampled using `voxel_down_sample` after merging. This is recommended post-processing after merging point clouds since it can relieve duplicated or over-densified points.

In [11]:
pcds = load_point_clouds(path_,voxel_size)
pcd_combined = o3d.geometry.PointCloud()
pcd_full = np.empty(shape=[0,4])
for point_id in range(len(pcds)):
    pcds[point_id].transform(pose_graph.nodes[point_id].pose)
    A = np.asarray(pcds[point_id].points)
    B = np.c_[ A, np.ones( A.shape[0] )*point_id ]
    pcd_full = np.r_[pcd_full,B]
    # pcd_combined += pcds[point_id]
# pcd_combined_down = pcd_combined.voxel_down_sample(voxel_size=voxel_size)
# o3d.io.write_point_cloud("multiway_registration.pcd", pcd_combined_down)
# o3d.visualization.draw_geometries([pcd_combined_down],
#                                   zoom=0.3412,
#                                   front=[0.4257, -0.2125, -0.8795],
#                                   lookat=[2.6172, 2.0475, 1.532],
#                                   up=[-0.0694, -0.9768, 0.2024])

D:/REGISTRATION DATA/External/JB-483 70ft EXT/tank_1\057.txt
D:/REGISTRATION DATA/External/JB-483 70ft EXT/tank_1\058.txt


In [ ]:
write_array2txt(
    path_ + '\pcd_57_58.txt',
    pcd_full)


<div class="alert alert-info">
    
**Note:**

Although this tutorial demonstrates multiway registration for point clouds, the same procedure can be applied to RGBD images. See [Make fragments](../reconstruction_system/make_fragments.rst) for an example.

</div>